In [1]:
import vtk

reader = vtk.vtkDICOMImageReader()
reader.SetDirectoryName("HeadNeck_A")  #### HERE YOU NEED TO ADD
reader.Update()

reader2 = vtk.vtkXMLImageDataReader()
reader2.SetFileName("aneurysm.vti")
reader2.update()

#this is your basic scene
renderWindow = vtk.vtkRenderWindow()
renderer = vtk.vtkRenderer()
renderWindow.AddRenderer(renderer)

#this is your basic interaction "provider"
iren = vtk.vtkRenderWindowInteractor()
iren.SetRenderWindow(renderWindow)

In [2]:
# Pixel Spacing
reader.GetPixelSpacing()

(0.6875, 0.6875, 5.000007629394531)

In [3]:
# Range of scalar values
print(reader2.GetOutput().GetPointData().GetScalars().GetRange())

(-1024.0, 1524.0)


In [4]:
# Let's modify the range so it starts from 0
math_vtk = vtk.vtkImageMathematics()
math_vtk.SetInputConnection(reader2.GetOutputPort())
math_vtk.SetOperationToAddConstant()
math_vtk.SetConstantC(1024.0)
math_vtk.Update()

In [5]:
print(math_vtk.GetOutput().GetPointData().GetScalars().GetRange())

(0.0, 2548.0)


In [6]:
# Set Threshold
thres_vtk = vtk.vtkImageThreshold()
thres_vtk.SetInputConnection(math_vtk.GetOutputPort())
# From which value onward do the values "match" as in stay
thres_vtk.ThresholdByUpper(1200.0)
# What should happen to the unmatched values?
thres_vtk.ReplaceOutOn()
# Replace them with 0
thres_vtk.SetOutValue(0.0)
# Inner values are not to be replaced
thres_vtk.ReplaceInOff()
thres_vtk.Update()

In [7]:
lut = vtk.vtkLookupTable()
lut.SetHueRange(0.0, 0.1)
lut.SetAlphaRange(0,1)
lut.Build()

In [8]:
planeWidget = vtk.vtkImagePlaneWidget()
planeWidget.SetInputConnection(reader2.GetOutputPort()) #### HERE YOU NEED TO ADD
planeWidget.SetInteractor(iren) #render window interactor
planeWidget.SetPlaneOrientationToZAxes()
planeWidget.PlaceWidget()
planeWidget.On() #enable the interaction

planeWidget4 = vtk.vtkImagePlaneWidget()
planeWidget4.SetInputConnection(thres_vtk.GetOutputPort()) #### HERE YOU NEED TO ADD
planeWidget4.SetInteractor(iren) #render window interactor
planeWidget4.SetPlaneOrientationToZAxes()
planeWidget4.PlaceWidget()
# This line adds the look up table defined above
planeWidget4.SetLookupTable(lut)
planeWidget4.On() #enable the interaction

def syncPlane(obj, event):
    planeWidget4.SetSliceIndex(obj.GetSliceIndex())
    
planeWidget.AddObserver("InteractionEvent", syncPlane)

iren.Initialize()
renderWindow.Render()
iren.Start()

In [ ]:
# There is still a problem with certain rotations getting rid of the colors. Probably z fighting, as I tried disabling culling, but it did not get rid of the artifact. 